# Cleaning Olympics Data

In [117]:
import pandas as pdy
results = pd.read_csv('results.csv', encoding='utf-8' , low_memory=False)

In [118]:
df = results.copy()
df.head()

,Games,Event,Team,Pos,Medal,As,athlete_id,NOC,Discipline,Nationality,Unnamed: 7
0,1912 Summer Olympics,"Singles, Men (Olympic)",NaN,=17,NaN,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN
1,1912 Summer Olympics,"Doubles, Men (Olympic)",Jean Montariol,DNS,NaN,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN
2,1920 Summer Olympics,"Singles, Men (Olympic)",NaN,=32,NaN,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN
3,1920 Summer Olympics,"Doubles, Mixed (Olympic)",Jeanne Vaussard,=8,NaN,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN
4,1920 Summer Olympics,"Doubles, Men (Olympic)",Jacques Brugnon,4,NaN,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN


### Cleaning to-do:
- [x] Rename AS to name
- [x] Split year and type from games
- [x] split out tied column from position
- [x] make non-numeric POSs NAN in Pos
- [x] Handle missing values
- [x] drop unnamed column
- [x] Reorder columns

In [120]:
df = df.rename(columns={'As': 'Name'})

In [121]:
df[['year', 'type']] = df['Games'].str.extract(r'(\d{4}) (Summer|Winter)', expand=True)
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

In [122]:
df.sort_values('year', ascending=False)

,Games,Event,Team,Pos,Medal,Name,athlete_id,NOC,Discipline,Nationality,Unnamed: 7,year,type
308407,2022 Winter Olympics,"Slalom, Women (Olympic)",NaN,24.0,NaN,Charlotta Säfvenberg,148986,NaN,NaN,NaN,NaN,2022,Winter
274176,2022 Winter Olympics,"Ice Hockey, Men (Olympic)",Latvia,11.0,NaN,Ronalds Ķēniņš,128130,LAT,Ice Hockey (Ice Hockey),NaN,NaN,2022,Winter
274171,2022 Winter Olympics,"Ice Hockey, Men (Olympic)",Latvia,DNS,NaN,Kristers Gudļevskis,128127,LAT,Ice Hockey (Ice Hockey),NaN,NaN,2022,Winter
274168,2022 Winter Olympics,"Ice Hockey, Men (Olympic)",Latvia,11,NaN,Ralfs Freibergs,128125,LAT,Ice Hockey (Ice Hockey),NaN,NaN,2022,Winter
274149,2022 Winter Olympics,"Four, Open (Olympic)",Latvia 1,5.0,NaN,Oskars Ķibermanis,128118,LAT,Bobsleigh (Bobsleigh),NaN,NaN,2022,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...
203177,1906 Intercalated Games,"Sabre, Individual, Men (Olympic)",NaN,5,NaN,Lóránt Mészáros,95189,HUN,Fencing,NaN,NaN,<NA>,NaN
203178,1906 Intercalated Games,"Sabre, Team, Men (Olympic)",Hungary,4,NaN,Lóránt Mészáros,95189,HUN,Fencing,NaN,NaN,<NA>,NaN
217586,1906 Intercalated Games,"Football, Men (Intercalated)",Athens,AC,NaN,Georgios Pantos,100811,GRE,Football (Football),NaN,NaN,<NA>,NaN
217587,1906 Intercalated Games,"Football, Men (Intercalated)",Athens,AC,NaN,Alexandros Kalafatis,100812,GRE,Football (Football),NaN,NaN,<NA>,NaN


In [123]:
df['tied'] = df['Pos'].str.contains("=", na=False) # create boolean print true if contains =
df['tied'] = df['tied'].astype('bool')

df['place'] = pd.to_numeric(df['Pos'].str.extract(r'(\d+)')[0], errors='coerce')
df['place'] = df['place'].fillna(0).astype('int64')

In [125]:
df['Medal'] = df["Medal"].map({"Bronze": 3, "Silver": 2, "Gold": 1})
df['Medal'] = df['Medal'].fillna('No Medal')

In [126]:
df['Team'] = df['Team'].fillna('Individual')

In [127]:
df.head()

,Games,Event,Team,Pos,Medal,Name,athlete_id,NOC,Discipline,Nationality,Unnamed: 7,year,type,tied,place
0,1912 Summer Olympics,"Singles, Men (Olympic)",Individual,=17,No Medal,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN,1912,Summer,True,17
1,1912 Summer Olympics,"Doubles, Men (Olympic)",Jean Montariol,DNS,No Medal,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN,1912,Summer,False,0
2,1920 Summer Olympics,"Singles, Men (Olympic)",Individual,=32,No Medal,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN,1920,Summer,True,32
3,1920 Summer Olympics,"Doubles, Mixed (Olympic)",Jeanne Vaussard,=8,No Medal,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN,1920,Summer,True,8
4,1920 Summer Olympics,"Doubles, Men (Olympic)",Jacques Brugnon,4,No Medal,Jean-François Blanchy,1,FRA,Tennis,NaN,NaN,1920,Summer,False,4


In [128]:
print(df.isnull().sum())

Games               0
Event               0
Team                0
Pos              1825
Medal               0
Name                0
athlete_id          0
NOC                 1
Discipline          1
Nationality    308327
Unnamed: 7     308408
year             2601
type             2601
tied                0
place               0
dtype: int64


In [129]:
df.columns

Index(['Games', 'Event', 'Team', 'Pos', 'Medal', 'Name', 'athlete_id', 'NOC',
       'Discipline', 'Nationality', 'Unnamed: 7', 'year', 'type', 'tied',
       'place'],
      dtype='object')

In [131]:
columns_to_keep = ['athlete_id', 'Name', 'year', 'Event', 'type', 'Discipline', 'Team', 'NOC', 'place', 'tied', 'Medal']
df = df[columns_to_keep]

In [132]:
df.to_csv('Results_clean.csv', index=False)

In [133]:
dd = pd.read_csv('Results_clean.csv')
dd.head()

,athlete_id,Name,year,Event,type,Discipline,Team,NOC,place,tied,Medal
0,1,Jean-François Blanchy,1912.0,"Singles, Men (Olympic)",Summer,Tennis,Individual,FRA,17,True,No Medal
1,1,Jean-François Blanchy,1912.0,"Doubles, Men (Olympic)",Summer,Tennis,Jean Montariol,FRA,0,False,No Medal
2,1,Jean-François Blanchy,1920.0,"Singles, Men (Olympic)",Summer,Tennis,Individual,FRA,32,True,No Medal
3,1,Jean-François Blanchy,1920.0,"Doubles, Mixed (Olympic)",Summer,Tennis,Jeanne Vaussard,FRA,8,True,No Medal
4,1,Jean-François Blanchy,1920.0,"Doubles, Men (Olympic)",Summer,Tennis,Jacques Brugnon,FRA,4,False,No Medal
